# PINO Baseline for PM2.5 Nowcasting (Murtuza Folder Only)

This notebook refactors the FNO2D baseline to a Physics-Informed Neural Operator (PINO) for PM2.5 forecasting, using engineered physical features, multi-objective loss, and baseline-compatible prediction export. All code and paths are strictly under the Murtuza folder for Kaggle compatibility.

In [ ]:
# Section 1: Kaggle Runtime Bootstrap (Murtuza-Only Paths)
import os, sys

# Kaggle input mounting
KAGGLE_SRC_DATASET = "murtuza-pm25-src"
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/khushisingh942004/aisehack"
MURTUZA_DIR = "/kaggle/input/datasets/sasidhar412/murtuza-pm25-src23/ANRF_AISEHack_Code/Murtuza"
DATA_DIR = KAGGLE_DATA_ROOT
CKPT_DIR = "/kaggle/temp"
OUT_DIR = "/kaggle/working"

if os.path.exists('/kaggle'):
    os.environ['AISEHACK_DATA'] = DATA_DIR
    SRC_ROOT = MURTUZA_DIR
else:
    SRC_ROOT = os.path.abspath('../Murtuza')
    DATA_DIR = os.path.abspath('../aisehack-theme-2')
    CKPT_DIR = os.path.abspath('./temp')
    OUT_DIR = os.path.abspath('./working')

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"AISEHACK_DATA: {os.environ.get('AISEHACK_DATA', 'not set')}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"CKPT_DIR: {CKPT_DIR}")
print(f"OUT_DIR: {OUT_DIR}")
assert 'Murtuza' in SRC_ROOT, "All imports/paths must resolve under Murtuza folder!"

In [ ]:
# Section 2: Repository Import, Seed, and Config Load
import random
import numpy as np
import torch
from src.config import load_config
from src.utils import seed_everything, print_device_info

cfg = load_config()
seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

# Ensure data path is correct for loader
cfg['paths']['data'] = DATA_DIR

print(f"Batch size: {cfg['training']['batch_size_train']}")
print(f"Learning rate: {cfg['training']['lr']}")
print(f"Forecast horizon: {cfg['time']['t_out']}")
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")

In [ ]:
# Section 3: Load ALL months, stack, add physical features
from src.data import add_physical_features, load_all_months, load_minmax_bounds
import numpy as np
import gc
import os

def stack_months(month_dicts, feature_list):
    """Stack list of month dicts into (N, C, T, H, W) float32 tensor."""
    T_min = min(d[feature_list[0]].shape[0] for d in month_dicts)
    return np.stack([
        np.stack([d[feat][:T_min] for feat in feature_list], axis=0)
        for d in month_dicts
    ], axis=0).astype(np.float32)

bounds = load_minmax_bounds(cfg)
all_months = cfg['data']['months']  # use the full dataset
print('Using months:', all_months)
train_data = load_all_months(cfg, all_months, bounds)
feature_list = cfg['features']['base']
tensor = stack_months(train_data, feature_list)

del train_data
gc.collect()
print(f"Base tensor shape (float32): {tensor.shape}  ({tensor.nbytes / 1e9:.2f} GB)")

features_dict = {name: idx for idx, name in enumerate(cfg['features']['base'])}
lat_long_path = os.path.join(cfg['paths']['data'], 'raw', 'lat_long.npy')
tensor = add_physical_features(tensor, features_dict, lat_long_path=lat_long_path)
gc.collect()

print(f"Tensor after physical features: {tensor.shape}  ({tensor.nbytes / 1e9:.2f} GB)")
print(f"Window-level validation split: {cfg['data'].get('val_split', 0.1):.2f}")

In [ ]:
# Section 4: (no-op) Lags and lat/lon are already built inside add_physical_features.
# Keeping this cell to avoid re-numbering but NOT allocating any extra arrays.
print("Engineered channels already in tensor — skipping duplicate computation.")
import gc; gc.collect()


In [ ]:
# Section 6: Patch Murtuza/src/model.py: FNO2D Input Expansion for PINO
from src.model import build_model

# Keep full time history in tensor for sliding-window supervision
print("Tensor shape before DataLoader windows:", tensor.shape)

# Model lift sees flattened (C * T_in) channels after reshape in forward
T_model = cfg['time'].get('t_in', 16)
input_channels = tensor.shape[1] * T_model
cfg['tensor_channels'] = input_channels
model = build_model(cfg)
print(f"Configured T_model: {T_model}")
print(f"Model input channels (C*T_model): {input_channels}")
print(f"Model type: {cfg['model']['type']}")

In [ ]:
# Section 7: Patch Murtuza/src/model.py: Circular/Reflect Padding Update
# Confirm model uses circular padding for Conv2d and spectral blocks
print('Model padding mode:', cfg.get('model', {}).get('padding_mode', 'circular'))

In [ ]:
# Section 8: Patch Murtuza/src/train.py: compute_physics_loss (Advection-Diffusion Residual)
from src.train import compute_physics_loss

# Example: compute physics loss for a batch
# physics_loss = compute_physics_loss(pred, xb, yb, cfg)
print('compute_physics_loss function ready for PINO training.')

In [ ]:
# Section 8: Patch Murtuza/src/train.py: compute_physics_loss (Advection-Diffusion Residual)
from src.train import compute_physics_loss

# Example: compute physics loss for a batch
# physics_loss = compute_physics_loss(pred, xb, yb, cfg)
print('compute_physics_loss function ready for PINO training.')

In [ ]:
# Section 10: Patch Murtuza/config.yaml: Add lambda_p, lambda_d, and PINO Hyperparameters
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")
print('PINO hyperparameters loaded from config.yaml.')

In [ ]:
# Section 10b: Create DataLoaders for Training and Validation
from src.data import make_dataloaders
train_dl, val_dl = make_dataloaders(cfg, tensor, bounds)
print('DataLoaders created for training and validation.')

In [ ]:
print("tensor.shape:", tensor.shape)

In [ ]:
# Optional diagnostics (do not mutate tensor before DataLoader creation)
print("Current tensor shape:", tensor.shape)
print("Expected training window (t_in):", cfg['time'].get('t_in', 16))
print("Expected forecast horizon (t_out):", cfg['time'].get('t_out', 16))

In [ ]:
# Section 11: Train Loop Execution with Logging (Data/Physics/Total Loss + RMSE)
from src.train import train

xb_dbg, yb_dbg = next(iter(train_dl))
print('Debug xb shape:', xb_dbg.shape)
print('Debug yb shape:', yb_dbg.shape)
assert yb_dbg.ndim == 4, f"Expected yb to be (B,H,W,T_out), got {yb_dbg.shape}"
assert yb_dbg.shape[-1] == cfg['time']['t_out'], f"Expected yb last dim T_out={cfg['time']['t_out']}, got {yb_dbg.shape}"

history = train(cfg, model, train_dl, val_dl, bounds=bounds)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ax = axes[0]
ax.plot(history['train_loss'], label='Train RMSE (norm)', alpha=0.8)
ax.plot(history['val_loss'],   label='Val RMSE (norm)',   alpha=0.9)
if 'val_persistence' in history:
    ax.plot(history['val_persistence'], label='Val persistence RMSE', alpha=0.8)
ax.set_xlabel('Epoch');  ax.set_ylabel('Normalized RMSE')
ax.set_title('Training Curves — Normalized Space')
ax.legend();  ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Section 12: Notebook Cell: Checkpoint Save/Load Compatible with exp_01_baseline.ipynb
import torch

# Save model checkpoint
torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'best_model.pt'))
print('Model checkpoint saved.')

# Load and sanity check
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, 'best_model.pt'), map_location=cfg['device']))
print('Model checkpoint loaded and verified.')

In [ ]:
# Section 13: Notebook Cell: 16-Hour Autoregressive Forecasting
from src.inference import run_inference

preds = run_inference(cfg, model, bounds)
print('Autoregressive 16-hour forecasting complete.')

In [ ]:
# Section 14: Notebook Cell: Final Prediction Export in Baseline-Compatible Format
np.save(os.path.join(OUT_DIR, 'preds.npy'), preds)
print(f'Predictions saved to {os.path.join(OUT_DIR, "preds.npy")}')